In [ ]:
import pandas as pd
from scipy.stats import wilcoxon

In [18]:
# 1. Cargar los archivos de resultados
depth = "0-1"
# Depth 0-1
if depth == "0-1":
    df_r2 = pd.read_csv('training_results/results_csv/results_in_0_1_R2_test.csv', index_col=0)
    df_rmse = pd.read_csv('training_results/results_csv/results_in_0_1_RMSE_test.csv', index_col=0)
# Depth 1-2
if depth == "1-2":
    df_r2 = pd.read_csv('training_results/results_csv/results_in_1_2_R2_test.csv', index_col=0)
    df_rmse = pd.read_csv('training_results/results_csv/results_in_1_2_RMSE_test.csv', index_col=0)
# Depth 2-3
if depth == "2-3":
    df_r2 = pd.read_csv('training_results/results_csv/results_in_2_3_R2_test.csv', index_col=0)
    df_rmse = pd.read_csv('training_results/results_csv/results_in_2_3_RMSE_test.csv', index_col=0)
# Depth 3-4
if depth == "3-4":
    df_r2 = pd.read_csv('training_results/results_csv/results_in_3_4_R2_test.csv', index_col=0)
    df_rmse = pd.read_csv('training_results/results_csv/results_in_3_4_RMSE_test.csv', index_col=0)


# 2. Función auxiliar para identificar la mejor configuración de forma segura
def get_best_index_safe(df, pattern, is_regex=False):
    # Buscar
    if is_regex:
        subset = df[df.index.str.contains(pattern, regex=True)]
    else:
        subset = df[df.index.str.contains(pattern)]
    
    # Si está vacío, devolver None
    if subset.empty:
        return None
        
    # Calcular el mejor (máximo promedio en este caso)
    return subset.mean(axis=1).idxmax()

# 3. Seleccionar los escenarios a comparar
print(f"Analizando Depth: {depth}")

# 1. C2X-Complex (Base)
best_c2xc = get_best_index_safe(df_r2, "C2X-Complex")

# 2. Otros (usando Regex para C2X estricto si es necesario)
# C2X(?!-Complex) busca "C2X" que NO vaya seguido de "-Complex"
best_c2x   = get_best_index_safe(df_r2, r"C2X(?!-Complex)", is_regex=True)
best_c2rcc = get_best_index_safe(df_r2, "C2RCC")
best_toa   = get_best_index_safe(df_r2, "TOA")

if best_c2xc is None:
    print("Error: No se encontró ningún dato para C2X-Complex (Base).")
else:
    print(f"Comparando {best_c2xc} (Base) contra otros procesadores...\n")

    # 4. Ejecutar el test de Wilcoxon para cada par
    comparisons = [
        ("C2RCC", best_c2rcc), 
        ("TOA", best_toa), 
        ("C2X", best_c2x)
    ]

    for name, row_id in comparisons:
        if row_id:
            # Test para R2 (Hipótesis: C2X-Complex > Otro)
            _, p_r2 = wilcoxon(df_r2.loc[best_c2xc], df_r2.loc[row_id], alternative='greater')
            
            # Test para RMSE (Hipótesis: C2X-Complex < Otro)
            _, p_rmse = wilcoxon(df_rmse.loc[best_c2xc], df_rmse.loc[row_id], alternative='less')
            
            sig_r2 = "(Significativo)" if p_r2 < 0.05 else "(No sig.)"
            sig_rmse = "(Significativo)" if p_rmse < 0.05 else "(No sig.)"
            
            print(f"--- C2X-Complex vs {name} ---")
            print(f"R2 p-valor: {p_r2:.4f} {sig_r2}")
            print(f"RMSE p-valor: {p_rmse:.4f} {sig_rmse}\n")
        else:
            print(f"Saltando comparación con {name}: No se encontraron datos.\n")

Analizando Depth: 0-1
Comparando C2X-Complex_rhow_9x9_depth_in_0_1 (Base) contra otros procesadores...

--- C2X-Complex vs C2RCC ---
R2 p-valor: 0.0098 (Significativo)
RMSE p-valor: 0.0107 (Significativo)

--- C2X-Complex vs TOA ---
R2 p-valor: 0.0029 (Significativo)
RMSE p-valor: 0.0098 (Significativo)

Saltando comparación con C2X: No se encontraron datos.

